# `06_non_bicycle_route_metrics`: Non-Bicycle-Route Cyclable Network Metrics per Spatial Unit

This notebook derives the following tables:
 
| Spatial unit | Metrics table | Descriptive stats |
|---|---|---|
| Municipality | `non_bicycle_route_metrics_by_municipality` | `non_bicycle_route_stats_by_municipality` |
| Province | `non_bicycle_route_metrics_by_province` | `non_bicycle_route_stats_by_province` |
| H3 cell | `non_bicycle_route_metrics_by_h3_cell` | `non_bicycle_route_stats_by_h3_cell` |
 
For each spatial unit, three metrics are computed: total cyclable way length outside the bicycle route network (`non_bicycle_route_length_km`), density (`non_bicycle_route_density_km_per_km2`), and length per 1,000 capita (`non_bicycle_route_km_per_1000_capita`). No network hierarchy breakdown is computed here: `non_bicycle_route_ways` are not members of any bicycle route relation and carry no network membership classification. These tables feed directly into `07_bicycle_route_vs_non_bicycle_route_metrics`.
 
---
 
## A note on what is being measured
 
This notebook examines the cyclable network that lies outside designated bicycle route relations, ways classified as cyclable under the conservative, tag-based definition established in `00_non_bicycle_route_ways`, but not members of any `route=bicycle` relation. These are ways where cycling is permitted or infrastructure is present, yet no responsible authority has designated them as part of a named, signed cycling route transcribed into OSM.

The distinction between `non_bicycle_route_ways` and `bicycle_route_m_ways_distinct` is important to frame correctly. `bicycle_route_m_ways_distinct` captures official designation, ways that an authority has included in a named, signed cycling route and a mapper has transcribed into OSM. `non_bicycle_route_ways` captures cyclable ways outside that designation, ways where cycling is explicitly permitted or infrastructure is present, but which do not form part of any named route relation. This includes protected cycle tracks, residential streets, rural roads, and shared-use paths alike. The two datasets measure different things and should not be interpreted as substitutes or ranked against each other. A long bicycle route network in a municipality reflects designation decisions; a long non-route cyclable network reflects the presence of cyclable ways that fall outside those decisions. Neither is a reliable indicator of the other, and a municipality with extensive non-route cyclable ways is not necessarily lacking in cycling provision, nor is one with a long designated route network necessarily well served by cycling infrastructure.
 
---
 
## Metrics
 
**Non-Bicycle-Route Length (`non_bicycle_route_length_km`)**: total length of cyclable ways outside the bicycle route network within a spatial unit. Subject to the same size-dependence as any absolute length measure; larger units will tend to show higher values.
 
**Non-Bicycle-Route Density (`non_bicycle_route_density_km_per_km2`)**: total length per square kilometre of land area. Enables size-normalised comparison across spatial units. Urban municipalities tend to score high on this metric because their street network is denser (more roads, cycleways, and residential streets per unit area) giving more opportunities for ways to satisfy the cyclability criteria, even where dedicated cycling infrastructure is limited.
 
**Non-Bicycle-Route Length per 1,000 Capita (`non_bicycle_route_km_per_1000_capita`)**: total length relative to the resident population, scaled to 1,000 inhabitants. Complements the density measure; rural municipalities with sparse populations but extensive cyclable road networks will show high per-capita values.
 
---
 
## Table of Contents
 
1. [Setup](#1-setup)
2. [Municipalities](#2-municipalities)
   - 2.1 [Compute metrics](#21-compute-metrics)
   - 2.2 [Descriptive statistics](#22-descriptive-statistics)
   - 2.3 [Visualisation](#23-visualisation)
3. [Provinces](#3-provinces)
   - 3.1 [Compute metrics](#31-compute-metrics)
   - 3.2 [Descriptive statistics](#32-descriptive-statistics)
   - 3.3 [Visualisation](#33-visualisation)
4. [H3 grid cells](#4-h3-grid-cells)
   - 4.1 [Compute metrics](#41-compute-metrics)
   - 4.2 [Descriptive statistics](#42-descriptive-statistics)
   - 4.3 [Visualisation](#43-visualisation)
5. [Derived tables](#5-derived-tables)
---

## 1. Setup

### Import relevant libraries

In [1]:
from IPython.utils import io
from lonboard import PolygonLayer, Map
from lonboard.colormap import apply_continuous_cmap
import ipywidgets as widgets
from IPython.display import display
import duckdb
import matplotlib.pyplot as plt

### Execute relevant notebooks and load variables into current session

In [2]:
with io.capture_output() as captured:
    %run /home/vbo226/03_boundaries_population.ipynb

non_bicycle_route_ways_per_municipality = duckdb.read_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_municipality.parquet"
)

non_bicycle_route_ways_per_h3_cell = duckdb.read_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_h3_cell.parquet"
)

non_bicycle_route_ways_per_province = duckdb.read_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_ways_per_province.parquet"
)

In [3]:
with io.capture_output() as captured:
    %run /home/vbo226/functions.ipynb

### Core functions


`calculate_metrics_per_spatial_unit` and `visualisation_minmax_norm_interactive` are defined in `06_bicycle_route_metrics` and reused here without modification. Running that notebook imports both functions into the current session, along with the Arrow-converted spatial unit tables (`municipalities_arrow`, `provinces_arrow`, `h3_cells_arrow`). No redefinition is needed.

In [4]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipalities_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

---

## 3. Municipalities


In [5]:
non_bicycle_route_metrics_by_municipality = calculate_metrics_per_spatial_unit(spatial_unit_table='municipalities_arrow', 
                                   ways_per_spatal_unit='non_bicycle_route_ways_per_municipality', 
                                    metric_prefix='non_bicycle_route',
                                   spatial_unit_code='municipality_code', 
                                   spatial_unit_name='municipality_name', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
non_bicycle_route_metrics_by_municipality_gdf = gpd.GeoDataFrame.from_arrow(non_bicycle_route_metrics_by_municipality.arrow())

### 3.2 Descriptive statistics

In [6]:
non_bicycle_route_stats_by_municipality = non_bicycle_route_metrics_by_municipality_gdf[['non_bicycle_route_length_km', 'non_bicycle_route_density_km_per_km2', 'non_bicycle_route_km_per_1000_capita']].describe()
non_bicycle_route_stats_by_municipality

,non_bicycle_route_length_km,non_bicycle_route_density_km_per_km2,non_bicycle_route_km_per_1000_capita
count,342.000000,342.000000,342.000000
mean,96.957026,1.272406,2.593035
std,91.555206,1.082297,3.408198
min,7.137000,0.061000,0.435000
25%,41.759000,0.520000,1.366250
50%,75.991000,0.917500,1.826000
75%,121.833750,1.663000,2.747750
max,886.283000,6.194000,39.231000


### 3.3 Visualisation


#### Interactive

In [7]:
map_non_bicycle_route_length_km_by_municipality  = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_municipality_gdf, 'non_bicycle_route_length_km')
map_non_bicycle_route_density_km_per_km2_by_municipality = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_municipality_gdf, 'non_bicycle_route_density_km_per_km2')
map_non_bicycle_route_km_per_1000_capita_by_municipality = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_municipality_gdf, 'non_bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total non bicycle route length (km)</h3>"),
        map_non_bicycle_route_length_km_by_municipality
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle route density (km per km2)</h3>"),
        map_non_bicycle_route_density_km_per_km2_by_municipality
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle_route_km_per_1000_capita</h3>"),
        map_non_bicycle_route_km_per_1000_capita_by_municipality
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

#### Static

## 4. Provinces
 
### 4.1 Compute metrics
 

In [9]:
non_bicycle_route_metrics_by_province = calculate_metrics_per_spatial_unit(spatial_unit_table='provinces_arrow', 
                                   ways_per_spatal_unit='non_bicycle_route_ways_per_province', 
                                   metric_prefix='non_bicycle_route',                                                                
                                   spatial_unit_code='province_code', 
                                   spatial_unit_name='province_name', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
non_bicycle_route_metrics_by_province_gdf = gpd.GeoDataFrame.from_arrow(non_bicycle_route_metrics_by_province.arrow())

### 4.2 Descriptive statistics

In [10]:
non_bicycle_route_stats_by_province = non_bicycle_route_metrics_by_province_gdf[['non_bicycle_route_length_km', 'non_bicycle_route_density_km_per_km2', 'non_bicycle_route_km_per_1000_capita']].describe()
non_bicycle_route_stats_by_province

,non_bicycle_route_length_km,non_bicycle_route_density_km_per_km2,non_bicycle_route_km_per_1000_capita
count,12.000000,12.000000,12.000000
mean,2763.275417,0.803917,2.114500
std,1739.437948,0.371926,0.621311
min,924.312000,0.249000,1.159000
25%,1416.962750,0.449000,1.794250
50%,2017.632500,0.893500,2.089500
75%,4333.593000,1.069500,2.398250
max,5839.516000,1.354000,3.132000


### 4.3 Visualisation

#### Interactive 

In [11]:
map_non_bicycle_route_length_km_by_province = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_province_gdf, 'non_bicycle_route_length_km')
map_non_bicycle_route_density_km_per_km2_by_province = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_province_gdf, 'non_bicycle_route_density_km_per_km2')
map_non_bicycle_route_km_per_1000_capita_by_province = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_province_gdf, 'non_bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total non bicycle route length (km)</h3>"),
        map_non_bicycle_route_length_km_by_province
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle route density (km per km2)</h3>"),
        map_non_bicycle_route_density_km_per_km2_by_province
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle_route_km_per_1000_capita</h3>"),
        map_non_bicycle_route_km_per_1000_capita_by_province
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

#### Static 

## 5. H3 grid cells

### 5.1 Compute metrics

In [12]:
non_bicycle_route_metrics_by_h3_cell = calculate_metrics_per_spatial_unit(spatial_unit_table='h3_cells_arrow', 
                                   ways_per_spatal_unit='non_bicycle_route_ways_per_h3_cell', 
                                   metric_prefix='non_bicycle_route',                                                                        
                                   spatial_unit_code='h3_index', 
                                   spatial_unit_geometry='geometry', 
                                   spatial_unit_areakm2='area_km2', 
                                   spatial_unit_population='population')

# Convert to GeoDataFrame
non_bicycle_route_metrics_by_h3_cell_gdf = gpd.GeoDataFrame.from_arrow(non_bicycle_route_metrics_by_h3_cell.arrow())

### 5.2 Descriptive statistics

In [13]:
non_bicycle_route_stats_by_h3_cell = non_bicycle_route_metrics_by_h3_cell_gdf[['non_bicycle_route_length_km', 'non_bicycle_route_density_km_per_km2', 'non_bicycle_route_km_per_1000_capita']].describe()
non_bicycle_route_stats_by_h3_cell

,non_bicycle_route_length_km,non_bicycle_route_density_km_per_km2,non_bicycle_route_km_per_1000_capita
count,66530.000000,66530.000000,49675.000000
mean,0.497975,0.797718,11.591799
std,0.958800,1.536334,83.042761
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.334000
75%,0.667750,1.068750,3.392000
max,11.257000,18.222000,5020.000000


### 5.3 Visualisation

#### Interactive

In [14]:
map_non_bicycle_route_length_km_by_h3_cell = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_h3_cell_gdf, 'non_bicycle_route_length_km')
map_non_bicycle_route_density_km_per_km2_by_h3_cell = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_h3_cell_gdf, 'non_bicycle_route_density_km_per_km2')
map_non_bicycle_route_km_per_1000_capita_by_h3_cell = visualization_minmax_norm_interactive(non_bicycle_route_metrics_by_h3_cell_gdf, 'non_bicycle_route_km_per_1000_capita')


map_ = widgets.HBox([
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Total non bicycle route length (km)</h3>"),
        map_non_bicycle_route_length_km_by_h3_cell
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle route density (km per km2)</h3>"),
        map_non_bicycle_route_density_km_per_km2_by_h3_cell
    ], layout=widgets.Layout(flex="1")),
    widgets.VBox([
        widgets.HTML("<h3 style='text-align:center;margin:0'>Non bicycle_route_km_per_1000_capita</h3>"),
        map_non_bicycle_route_km_per_1000_capita_by_h3_cell
    ], layout=widgets.Layout(flex="1"))
])

display(map_)

/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


#### Static

### Export

In [15]:
from pathlib import Path

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

non_bicycle_route_metrics_by_municipality.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_metrics_by_municipality.parquet"
)

non_bicycle_route_stats_by_municipality.to_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_stats_by_municipality.parquet"
)


non_bicycle_route_metrics_by_province.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_metrics_by_province.parquet"
)

non_bicycle_route_stats_by_province.to_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_stats_by_province.parquet"
)

non_bicycle_route_metrics_by_h3_cell.write_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_metrics_by_h3_cell.parquet"
)

non_bicycle_route_stats_by_h3_cell.to_parquet(
    "/local/data/vbo226/cache/non_bicycle_route_stats_by_h3_cell.parquet"
)